# NB_lab_test_trends

Builds `dbx_caresync.gold.lab_test_trends` — one row per test type per calendar month with test volumes, abnormal/critical rates, and month-over-month trend metrics.

In [0]:
%sql
CREATE OR REPLACE TABLE dbx_caresync.gold.lab_test_trends
AS
WITH monthly_agg AS (
  SELECT
    lr.test_name,
    YEAR(lr.test_date)                                              AS year,
    MONTH(lr.test_date)                                             AS month,
    COUNT(*)                                                        AS total_tests,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)       AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)       AS critical_test_count
  FROM dbx_caresync.silver.lab_results AS lr
  GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
),
with_rates AS (
  SELECT
    ma.test_name,
    ma.year,
    ma.month,
    c.month_name,
    c.quarter_name,
    ma.total_tests,
    ma.abnormal_test_count,
    ma.critical_test_count,
    ROUND(ma.abnormal_test_count / ma.total_tests * 100, 2)         AS abnormal_rate_pct,
    ROUND(ma.critical_test_count / ma.total_tests * 100, 2)         AS critical_rate_pct
  FROM monthly_agg                                 AS ma
  LEFT JOIN dbx_caresync.silver.calendar            AS c
         ON c.date_key = CAST(CONCAT(
              LPAD(ma.year, 4, '0'),
              LPAD(ma.month, 2, '0'),
              '01'
            ) AS INT)
)
SELECT
  test_name,
  year,
  month,
  month_name,
  quarter_name,
  total_tests,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month)  AS prev_month_abnormal_rate_pct,
  ROUND(
    abnormal_rate_pct
    - LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month),
    2
  )                                                                          AS abnormal_rate_change_pct
FROM with_rates
ORDER BY test_name, year, month;

In [0]:
updated_df = spark.table("dbx_caresync.gold.lab_test_trends")
records_count = updated_df.count()
dbutils.notebook.exit(str(records_count))